<a href="https://colab.research.google.com/github/tharujayasinghe163/Statistical-Learning-e22163/blob/main/Assignment_7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Deriving the Marginal Density

By the law of total probability, the marginal density $p(x_i)$ is obtained by summing the joint distribution $p(x_i, C_i=k)$ over all possible states of the latent variable $C_i$:

$$p(x_i) = \sum_{k=1}^K p(x_i, C_i=k) = \sum_{k=1}^K P(C_i=k)\,p(x_i \mid C_i=k)$$

Substituting the prior cluster membership probability $P(C_i=k) = \phi_k$ and the conditional Gaussian distribution $p(x_i \mid C_i=k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$, we arrive at:

$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$

**Why it is called a Gaussian mixture density:**
This expression is a linear combination of $K$ distinct probability density functions, mixed together using the non-negative scalar weights $\phi_k$. Because each individual density in the sum is a Gaussian distribution and the weights sum to 1 ($\sum \phi_k = 1$), it represents a probabilistic blend or "mixture" of Gaussians.

---

### 2. Deriving the Posterior Cluster Probability

By Bayes' rule, the conditional probability of a discrete latent event $C_i=k$ given an observed continuous value $X_i=x_i$ is:

$$P(C_i=k \mid X_i=x_i) = \frac{p(X_i=x_i, C_i=k)}{p(x_i)} = \frac{P(X_i=x_i \mid C_i=k)P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i \mid C_j=j)P(C_i=j)}$$

Substituting the given component densities and priors directly yields the responsibility $\gamma_{ik}$:

$$\gamma_{ik} = P(C_i=k \mid X_i=x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$

**Interpretation of $\gamma_{ik}$:**
Before looking at the data point $x_i$, our initial belief that the point belongs to cluster $k$ is its prior probability $\phi_k$. Once we observe the exact spatial location $x_i$, the likelihood term $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ scales this prior based on cluster proximity. The normalized result $\gamma_{ik}$ updates our belief, serving as the **posterior probability** that cluster $k$ was the specific generative source behind $x_i$.

---

### 3. One-Hot Encoding of the Latent Cluster Variable

Since $Z_{ik}$ is a binary indicator random variable ($Z_{ik} \in \{0, 1\}$), its conditional expectation is fundamentally equal to the probability of it evaluating to 1:

$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = 1 \cdot P(Z_{ik}=1 \mid X_i=x_i) + 0 \cdot P(Z_{ik}=0 \mid X_i=x_i) = P(C_i=k \mid X_i=x_i)$$

Using the responsibility notation defined in Part 2, $\mathbb{E}[Z_{ik} \mid X_i=x_i] = \gamma_{ik}$. Applying this component-wise across the full vector $Z_i$:

$$\mathbb{E}[Z_i \mid X_i=x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i=x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i=x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

**Conclusion:**
Instead of making a discrete choice, a soft assignment maps data points to a continuous distribution across clusters. Mathematically, this soft mapping is exactly the conditional expectation vector $\mathbb{E}[Z_i \mid X_i=x_i]$, where each entry represents a fractional probability matching the model's posterior confidence.

---

### 4. From Soft Assignment to Hard Clustering

* **Soft Clustering:** Assigns each observation $x_i$ a continuous distribution vector across all $K$ clusters via $\mathbb{E}[Z_i \mid X_i=x_i]$. A point can concurrently hold fractional memberships (e.g., $50\%$ in cluster 1 and $50\%$ in cluster 2), preserving a mathematical measure of geometric ambiguity.
* **Hard Clustering:** Strips away this ambiguity by mapping each observation to exactly one cluster using the Maximum A Posteriori (MAP) decision rule: $\widehat{C}_i = \operatorname{arg\,max}_k \gamma_{ik}$. This converts continuous probabilities into a single, deterministic index, discarding uncertainty information for points near cluster boundaries.

---

### 5. Conditional Expectation of the Observation Given the Cluster

Given the conditional distribution $X_i \mid C_i=k \sim \mathscr{N}(\mu_k, \Sigma_k)$, the conditional expectation corresponds directly to the mean parameter of that specific multivariate normal distribution:

$$\mathbb{E}[X_i \mid C_i=k] = \int_{\mathbb{R}^d} x_i \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i = \mu_k$$

**Interpretation of $\mu_k$:**
The vector $\mu_k$ represents the center of cluster $k$ because it defines the center of mass (the balance point) of its probability density cloud in the $d$-dimensional space $\mathbb{R}^d$.

**Comparing the Two Expectations:**

* $\mathbb{E}[Z_i \mid X_i=x_i]$ operates on the *latent space*. It takes a **known spatial position** $x_i$ and outputs a vector of probabilities showing how that position splits across clusters.
* $\mathbb{E}[X_i \mid C_i=k]$ operates on the *feature space*. It assumes a **known cluster identity** $k$ and outputs the expected coordinates ($\mu_k$) of an observation drawn from that group.

---

### 6. The Complete-Data Likelihood

Given the complete-data likelihood:

$$p(x_1,\dots,x_n,z_1,\dots,z_n) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

Taking the natural logarithm transforms the nested products into nested sums, while the exponents $z_{ik}$ pull down as linear multipliers:

$$\ell_c = \log p(x_1,\dots,x_n,z_1,\dots,z_n) = \sum_{i=1}^n \sum_{k=1}^K \log \left( \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

**Why it is easy to maximize:**
If the latent indicators $z_{ik}$ were known constants, the optimization problem cleanly decouples across clusters. To find the optimal parameters for cluster $k$, we would only need to compute the analytical maximum likelihood estimates (MLE) using the subset of data points assigned to that cluster ($z_{ik}=1$). This removes the complex sum-inside-the-log term found in standard marginal GMM likelihoods.

---

### 7. The EM Interpretation

The Expectation step (E-step) handles unobserved data by computing the conditional expectation of the complete-data log-likelihood with respect to the latent variables $Z_i$, conditioned on the observed data $X$ and the current parameter estimates $\Theta^{(t)}$:

$$Q(\Theta \mid \Theta^{(t)}) = \mathbb{E}_{Z \mid X, \Theta^{(t)}} [\ell_c]$$

Because $\ell_c$ is linear with respect to the indicators $z_{ik}$, the expectation operator passes directly inside the summation onto the random variables:

$$Q(\Theta \mid \Theta^{(t)}) = \sum_{i=1}^n \sum_{k=1}^K \mathbb{E}[Z_{ik} \mid X_i=x_i, \Theta^{(t)}] \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

From our proof in Part 3, $\mathbb{E}[Z_{ik} \mid X_i=x_i, \Theta^{(t)}] = \gamma_{ik}^{(t)}$. Substituting this gives:

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

**Interpretation of the E-step:**
The E-step can be viewed as a conditional update of cluster membership probabilities. It uses our current parameter estimates to evaluate how well each cluster explains each data point, outputting a new set of continuous responsibility weights $\gamma_{ik}$ for the entire dataset.

---

### 8. Parameter Updates

To derive the M-step updates, we maximize the objective function $Q$ with respect to each parameter individually.

#### Updating $\phi_k$

We maximize $Q$ subject to the equality constraint $\sum_{k=1}^K \phi_k = 1$ using a Lagrange multiplier $\lambda$:

$$\mathcal{L}(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k + \lambda \left( 1 - \sum_{k=1}^K \phi_k \right)$$

Taking the partial derivative with respect to $\phi_k$ and setting it to zero:

$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda}$$

Summing both sides over all $k$ allows us to solve for $\lambda$:

$$\sum_{k=1}^K \phi_k = \sum_{k=1}^K \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda} \implies 1 = \frac{1}{\lambda} \sum_{i=1}^n \underbrace{\sum_{k=1}^K \gamma_{ik}}_{1} \implies 1 = \frac{n}{\lambda} \implies \lambda = n$$

Defining the effective number of points assigned to cluster $k$ as $N_k = \sum_{i=1}^n \gamma_{ik}$, we get:

$$\phi_k^{\text{new}} = \frac{N_k}{n}$$

#### Updating $\mu_k$

Expanding the log of the multivariate normal density shows the terms that depend on $\mu_k$:

$$\log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) = -\frac{1}{2}(x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) + \text{const}$$

Taking the partial derivative of $Q$ with respect to $\mu_k$:

$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^n \gamma_{ik} \left( \Sigma_k^{-1} (x_i - \mu_k) \right) = 0$$

Since $\Sigma_k^{-1}$ is a non-singular matrix, we can multiply by $\Sigma_k$ to eliminate it:

$$\sum_{i=1}^n \gamma_{ik} (x_i - \mu_k) = 0 \implies \sum_{i=1}^n \gamma_{ik} x_i = \left(\sum_{i=1}^n \gamma_{ik}\right) \mu_k \implies \mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$$

#### Updating $\Sigma_k$

Optimizing with respect to the precision matrix $\Sigma_k^{-1}$ yields standard weighted covariance updates:

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$

**How $\gamma_{ik}$ acts as a fractional weight:**
In standard maximum likelihood estimation, every data point contributes equally (weight = 1). In the GMM updates, the responsibility $\gamma_{ik}$ scales each data point's contribution. If a point has a high probability of belonging to cluster $k$ ($\gamma_{ik} \approx 1$), it heavily influences that cluster's new mean and covariance. If $\gamma_{ik} \approx 0$, its impact on cluster $k$ is negligible.

---

### 9. Interpretation

Gaussian mixture model (GMM) clustering can be viewed as an iterative process of conditional updates. The mixture weight $\phi_k$ acts as our prior probability for cluster $k$, while the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ determines the likelihood, measuring how compatible a given data point $x_i$ is with that cluster's current spatial center and spread. By combining these terms through Bayes' rule, the model computes the responsibility $\gamma_{ik}$, which represents the updated posterior probability of cluster membership after observing $x_i$. Collected across all groups, these weights form the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_i]$. During the Maximization step (M-step), the algorithm uses these posterior probabilities as continuous weights to update the cluster parameters, pulling the cluster centers and shapes toward the data points they explain best.

Ultimately, Gaussian mixture clustering is a robust probabilistic framework built entirely on the conditional expectations of latent cluster membership variables.

---

### 10. Computational Simulation and Out-of-Sample Validation



In [ ]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.express as px
import plotly.graph_objects as go

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=self.n_components,
            random_state=self.random_state,
            init_params='kmeans'
        )
        self.X_train_scaled = None
        self.X_test_scaled = None
        self.features = None

    def load_and_preprocess(self, url):
        """Loads the dataset, selects specific features, drops missing rows, and splits/scales."""
        print("Fetching dataset...")
        df = pd.read_csv(url)

        # Select two continuous features for meaningful 2D visualization
        self.features = ['PURCHASES', 'CREDIT_LIMIT']
        df_sub = df[self.features].dropna()

        X = df_sub.values

        # 80/20 Train/Test Split
        X_train, X_test = train_test_split(X, test_size=0.2, random_state=self.random_state)

        # Fit scaler on training data only to avoid data leakage
        self.X_train_scaled = self.scaler.fit_transform(X_train)
        self.X_test_scaled = self.scaler.transform(X_test)
        print(f"Data processing complete. Train shape: {self.X_train_scaled.shape}, Test shape: {self.X_test_scaled.shape}")

    def fit(self):
        """Fits the GMM model on scaled training data and logs convergence metrics."""
        self.model.fit(self.X_train_scaled)
        print("\n--- EM Execution Summary ---")
        print(f"Model Converged: {self.model.converged_}")
        print(f"Iterations Required: {self.model.n_iter_}")

    def evaluate(self):
        """Computes average log-likelihood scores for train and out-of-sample data."""
        train_score = self.model.score(self.X_train_scaled)
        test_score = self.model.score(self.X_test_scaled)
        print("\n--- Out-of-Sample Performance Summary ---")
        print(f"Average Log-Likelihood (Train Set): {train_score:.4f}")
        print(f"Average Log-Likelihood (Test Set Object):  {test_score:.4f}")
        return test_score

    def plot_density_heatmap(self):
        """Figure 1: 2D Density Heatmap with marginal distributions."""
        fig = px.density_heatmap(
            x=self.X_train_scaled[:, 0],
            y=self.X_train_scaled[:, 1],
            marginal_x="histogram",
            marginal_y="histogram",
            labels={'x': self.features[0] + ' (Scaled)', 'y': self.features[1] + ' (Scaled)'},
            title="Figure 1: Empirical 2D Density Heatmap of Raw Training Data",
            color_continuous_scale="Viridis"
        )
        fig.update_layout(template="plotly_white")
        return fig

    def _generate_contour_mesh(self):
        """Helper to compute responsibilities across a fine grid boundary map."""
        x_min, x_max = self.X_train_scaled[:, 0].min() - 0.5, self.X_train_scaled[:, 0].max() + 0.5
        y_min, y_max = self.X_train_scaled[:, 1].min() - 0.5, self.X_train_scaled[:, 1].max() + 0.5

        x_grid = np.linspace(x_min, x_max, 200)
        y_grid = np.linspace(y_min, y_max, 200)
        xx, yy = np.meshgrid(x_grid, y_grid)
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # Compute responsibilities gamma_ik for every point in the mesh
        responsibilities = self.model.predict_proba(grid_points)
        max_resp = responsibilities.max(axis=1).reshape(xx.shape)

        return x_grid, y_grid, max_resp

    def plot_assignments(self, dataset_type="train"):
        """Figure 2 & 3: Overlays data points on top of the continuous responsibility contour map."""
        x_grid, y_grid, max_resp = self._generate_contour_mesh()

        if dataset_type == "train":
            data_points = self.X_train_scaled
            labels = self.model.predict(self.X_train_scaled)
            title = "Figure 2: Training Assignment Plot over Continuous Confidence Boundaries"
        else:
            data_points = self.X_test_scaled
            labels = self.model.predict(self.X_test_scaled)
            title = "Figure 3: Out-of-Sample Test Assignment Plot exposing Edge Ambiguities"

        fig = go.Figure()

        # Add background continuous contour plot representing max posterior probability
        fig.add_trace(go.Contour(
            x=x_grid, y=y_grid, z=max_resp,
            colorscale="Electric",
            contours_coloring="heatmap",
            showscale=True,
            colorbar=dict(title="Max Posterior<br>Responsibility (γ)"),
            opacity=0.4,
            hoverinfo="skip"
        ))

        # Scatter overlay of hard assignments
        for cluster_idx in range(self.n_components):
            cluster_mask = (labels == cluster_idx)
            fig.add_trace(go.Scatter(
                x=data_points[cluster_mask, 0],
                y=data_points[cluster_mask, 1],
                mode='markers',
                marker=dict(size=5, line=dict(width=0.5, color='DarkSlateGrey')),
                name=f"Hard Cluster {cluster_idx}"
            ))

        fig.update_layout(
            title=title,
            xaxis_title=self.features[0] + " (Standardized)",
            yaxis_title=self.features[1] + " (Standardized)",
            template="plotly_white",
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
        )
        return fig

# --- Execution block ---
if __name__ == "__main__":
    # Direct public data link to Kaggle CC GENERAL data via data-mirror
    data_url = "https://raw.githubusercontent.com/vickyramprakash/Credit-Card-Customer-Data-Clustering/master/CC%20GENERAL.csv"

    segmenter = GMMFinancialSegmenter(n_components=3)
    segmenter.load_and_preprocess(data_url)
    segmenter.fit()
    segmenter.evaluate()

    # Generate the plots (In a Jupyter notebook context, use .show() to render interactively)
    fig1 = segmenter.plot_density_heatmap()
    fig2 = segmenter.plot_assignments(dataset_type="train")
    fig3 = segmenter.plot_assignments(dataset_type="test")

    print("\n[Plots generated successfully! Use fig.show() within your local notebook environment to view them.]")